[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Python_File_IO.ipynb)

# File I/O and Data Formats

Read and write text, CSV, JSON, JSON Lines, and Parquet — the formats you will encounter in every data pipeline. Plus working with APIs and the filesystem.

**Concept chapter:** [File I/O](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/04_File_IO.md)

**Community:** [Delivery Momentum on Skool](https://www.skool.com/deliverymomentum)

## 1. Text Files — Read and Write

The `with` statement guarantees the file is closed even if an error occurs. Always use it.

In [ ]:
# WHAT: Write and read a text log file.
# WHY: Pipeline logs, SQL scripts, and config files are all text.
# The 'with' statement is Python's try-with-resources.

from pathlib import Path

log_dir = Path("data")
log_dir.mkdir(exist_ok=True)
log_file = log_dir / "pipeline.log"

# Write lines to a file
log_lines = [
    "2026-04-04 10:00:00 INFO  Pipeline started",
    "2026-04-04 10:00:05 INFO  Extracted 1,250,000 rows",
    "2026-04-04 10:00:47 INFO  Transform complete",
    "2026-04-04 10:00:48 ERROR Duplicate key violation on row 42",
    "2026-04-04 10:00:50 INFO  Pipeline finished with errors"
]

with open(log_file, "w") as f:
    for line in log_lines:
        f.write(line + "\n")

print(f"Wrote {len(log_lines)} lines to {log_file}")

# Read the file back
with open(log_file, "r") as f:
    content = f.readlines()

# Filter for errors
errors = [line.strip() for line in content if "ERROR" in line]
print(f"Errors found: {len(errors)}")
for err in errors:
    print(f"  {err}")

# You Should See:
# Wrote 5 lines to data/pipeline.log
# Errors found: 1
#   2026-04-04 10:00:48 ERROR Duplicate key violation on row 42

## 2. CSV Files

CSV is the interchange format for tabular data. Two approaches: the `csv` module (lightweight) and `pandas` (powerful).

In [ ]:
# WHAT: Write and read CSV with the csv module.
# WHY: The csv module is in the standard library — no pip install.
# Use it for simple scripts; use pandas for analysis.

import csv
from pathlib import Path

csv_file = Path("data/calls_sample.csv")

# Write call center data
calls = [
    {"call_id": "C001", "agent": "Alice", "duration_sec": 180, "resolved": "true"},
    {"call_id": "C002", "agent": "Bob", "duration_sec": 420, "resolved": "false"},
    {"call_id": "C003", "agent": "Alice", "duration_sec": 95, "resolved": "true"},
    {"call_id": "C004", "agent": "Carol", "duration_sec": 310, "resolved": "true"}
]

with open(csv_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=calls[0].keys())
    writer.writeheader()
    writer.writerows(calls)

print(f"Wrote {len(calls)} rows to {csv_file}")

# Read it back
with open(csv_file, "r") as f:
    reader = csv.DictReader(f)
    loaded = list(reader)

# Note: ALL values come back as strings
print(f"\nLoaded {len(loaded)} rows")
print(f"First row: {loaded[0]}")
print(f"Duration type: {type(loaded[0]['duration_sec'])}")

# Common mistake: csv.DictReader returns strings, not typed values.
# You must convert: int(row['duration_sec'])

# You Should See:
# Wrote 4 rows to data/calls_sample.csv
# Loaded 4 rows
# First row: {'call_id': 'C001', 'agent': 'Alice', 'duration_sec': '180', 'resolved': 'true'}
# Duration type: <class 'str'>

In [ ]:
# WHAT: Read CSV with pandas — automatic type inference.
# WHY: Pandas handles type conversion, missing values, and
# encoding issues automatically. Use it for any analysis work.

import pandas as pd

df = pd.read_csv("data/calls_sample.csv")

print("Shape:", df.shape)
print("\nTypes:")
print(df.dtypes)
print("\nData:")
print(df.to_string(index=False))

# Note: pandas inferred duration_sec as int64 automatically

# You Should See:
# Shape: (4, 4)
# Types:
# call_id         object
# agent           object
# duration_sec     int64
# resolved        object

## 3. JSON Files

JSON is the format for API responses, config files, and NoSQL data. Python's `json` module handles it natively.

In [ ]:
# WHAT: Write and read JSON with the json module.
# WHY: API responses, config files, and MongoDB documents
# are all JSON. This is the data interchange format of the web.

import json
from pathlib import Path

json_file = Path("data/model_config.json")

config = {
    "model_name": "xgboost-churn-v2",
    "hyperparameters": {
        "learning_rate": 0.01,
        "max_depth": 6,
        "n_estimators": 500
    },
    "features": ["age", "income", "tenure_months", "credit_score"],
    "target": "churned",
    "trained_at": "2026-04-04T10:00:00Z"
}

# Write with pretty printing
with open(json_file, "w") as f:
    json.dump(config, f, indent=2)

print(f"Wrote config to {json_file}")

# Read it back
with open(json_file, "r") as f:
    loaded = json.load(f)

print(f"Model: {loaded['model_name']}")
print(f"LR: {loaded['hyperparameters']['learning_rate']}")
print(f"Features: {loaded['features']}")

# You Should See:
# Wrote config to data/model_config.json
# Model: xgboost-churn-v2
# LR: 0.01
# Features: ['age', 'income', 'tenure_months', 'credit_score']

In [ ]:
# WHAT: Write and read JSON Lines (one JSON object per line).
# WHY: JSON Lines is the standard for streaming data, log processing,
# and BigQuery/Spark ingestion. Each line is independently parseable.

import json
from pathlib import Path

jsonl_file = Path("data/events.jsonl")

events = [
    {"timestamp": "2026-04-04T10:00:00Z", "event": "model_loaded", "model": "bert-v2"},
    {"timestamp": "2026-04-04T10:00:05Z", "event": "prediction", "score": 0.95},
    {"timestamp": "2026-04-04T10:00:06Z", "event": "prediction", "score": 0.42},
    {"timestamp": "2026-04-04T10:00:10Z", "event": "model_unloaded", "model": "bert-v2"}
]

# Write — one JSON object per line, no enclosing array
with open(jsonl_file, "w") as f:
    for event in events:
        f.write(json.dumps(event) + "\n")

# Read — parse each line independently
with open(jsonl_file, "r") as f:
    loaded = [json.loads(line) for line in f]

print(f"Loaded {len(loaded)} events")
for e in loaded:
    print(f"  {e['timestamp']}: {e['event']}")

# You Should See:
# Loaded 4 events
#   2026-04-04T10:00:00Z: model_loaded
#   2026-04-04T10:00:05Z: prediction
#   2026-04-04T10:00:06Z: prediction
#   2026-04-04T10:00:10Z: model_unloaded

## 4. Parquet Files

Parquet is columnar, compressed, and typed — the production format for data lakes. It is 5-10x smaller than CSV and 10-100x faster to query.

In [ ]:
# WHAT: Write and read Parquet with pandas.
# WHY: Parquet is the standard for S3 data lakes, Spark, and
# BigQuery. Types are preserved (no string-to-int issues like CSV).

import pandas as pd
from pathlib import Path

parquet_file = Path("data/calls.parquet")

# Create a DataFrame
calls_df = pd.DataFrame([
    {"call_id": "C001", "agent": "Alice", "duration_sec": 180, "resolved": True},
    {"call_id": "C002", "agent": "Bob", "duration_sec": 420, "resolved": False},
    {"call_id": "C003", "agent": "Alice", "duration_sec": 95, "resolved": True},
    {"call_id": "C004", "agent": "Carol", "duration_sec": 310, "resolved": True}
])

# Write to Parquet
calls_df.to_parquet(parquet_file, index=False)
print(f"Wrote {len(calls_df)} rows to {parquet_file}")

# Read back — types are preserved!
loaded = pd.read_parquet(parquet_file)
print(f"\nTypes from Parquet (notice: bool, not object):")
print(loaded.dtypes)

# Compare file sizes
csv_size = Path("data/calls_sample.csv").stat().st_size
pq_size = parquet_file.stat().st_size
print(f"\nCSV size:     {csv_size:,} bytes")
print(f"Parquet size: {pq_size:,} bytes")

# You Should See:
# Wrote 4 rows to data/calls.parquet
# Types from Parquet (notice: bool, not object):
# call_id         object
# agent           object
# duration_sec     int64
# resolved          bool

In [ ]:
# WHAT: Read only specific columns from Parquet.
# WHY: Parquet is columnar — you can read 3 columns from a
# 100-column file without loading the other 97. Massive speedup.

import pandas as pd

# Read only the columns you need
subset = pd.read_parquet("data/calls.parquet", columns=["call_id", "duration_sec"])
print(f"Columns loaded: {list(subset.columns)}")
print(subset.to_string(index=False))

# You Should See:
# Columns loaded: ['call_id', 'duration_sec']
# call_id  duration_sec
#    C001           180
#    C002           420
#    C003            95
#    C004           310

## 5. Working with APIs

The `requests` library is the standard for HTTP calls. API responses come back as JSON.

In [ ]:
# WHAT: Make a GET request and parse the JSON response.
# WHY: ML services, data APIs, and cloud services all use
# REST APIs. This is how you call them.

import requests

# Public API — no auth required
url = "https://httpbin.org/json"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # raises exception for 4xx/5xx
    
    data = response.json()  # parse JSON body into a dict
    
    print(f"Status: {response.status_code}")
    print(f"Content type: {response.headers.get('content-type')}")
    print(f"Response keys: {list(data.keys())}")
    
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

# You Should See:
# Status: 200
# Content type: application/json
# Response keys: ['slideshow']

In [ ]:
# WHAT: Download a file from a URL.
# WHY: Datasets, model weights, and config files often need
# to be downloaded at pipeline startup.

import requests
from pathlib import Path

url = "https://httpbin.org/robots.txt"
output_path = Path("data/downloaded.txt")

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    # Write binary content (works for any file type)
    with open(output_path, "wb") as f:
        f.write(response.content)
    
    print(f"Downloaded {len(response.content)} bytes to {output_path}")
    
    # Verify content
    print(f"Content preview: {response.text[:100]}")

except requests.exceptions.RequestException as e:
    print(f"Download failed: {e}")

# You Should See:
# Downloaded N bytes to data/downloaded.txt
# Content preview: (first 100 chars of the file)

In [ ]:
# WHAT: Read configuration from environment variables.
# WHY: Secrets (API keys, passwords) must NEVER be in code.
# os.getenv reads from the shell environment and provides defaults.

import os

# os.getenv returns None (or your default) if the variable is not set
api_key = os.getenv("OPENAI_API_KEY", "not-set")
db_host = os.getenv("DATABASE_HOST", "localhost")
debug = os.getenv("DEBUG", "false").lower() == "true"

print(f"API key: {'***' + api_key[-4:] if api_key != 'not-set' else 'not-set'}")
print(f"DB host: {db_host}")
print(f"Debug:   {debug}")

# Common mistake: os.environ['MISSING_VAR'] raises KeyError.
# Always use os.getenv('VAR', default) for optional vars.

# You Should See:
# API key: not-set (unless you have OPENAI_API_KEY set)
# DB host: localhost
# Debug:   False

## 6. Filesystem Operations with pathlib

`pathlib` is the modern way to work with file paths. It replaces `os.path.join()`, `os.listdir()`, and `os.makedirs()` with a cleaner object-oriented API.

In [ ]:
# WHAT: List files, check existence, and get file info with pathlib.
# WHY: Pipeline startup checks: does the input exist? How old is it?
# What files are in the landing zone?

from pathlib import Path
from datetime import datetime

data_dir = Path("data")

# List all files in a directory
print("Files in data/:")
for f in sorted(data_dir.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:30s} {size_kb:>8.1f} KB  ({f.suffix})")

# Glob for specific patterns
json_files = list(data_dir.glob("*.json"))
print(f"\nJSON files: {[f.name for f in json_files]}")

# Check existence before reading
target = data_dir / "calls.parquet"
if target.exists():
    print(f"\n{target.name} exists ({target.stat().st_size:,} bytes)")
else:
    print(f"\n{target.name} not found — need to generate it")

# You Should See:
# Files in data/:
#   calls.parquet                     ... KB  (.parquet)
#   calls_sample.csv                  ... KB  (.csv)
#   ... (all files listed with sizes)

In [ ]:
# WHAT: Build file paths safely with pathlib.
# WHY: Hard-coded paths break across operating systems.
# pathlib handles / vs \ automatically.

from pathlib import Path

# Build paths with / operator — works on Windows and Unix
base = Path("data")
output = base / "output" / "2026" / "04" / "results.parquet"

print(f"Full path: {output}")
print(f"Parent:    {output.parent}")
print(f"Stem:      {output.stem}")       # filename without extension
print(f"Suffix:    {output.suffix}")     # extension
print(f"Name:      {output.name}")       # filename with extension

# Create parent directories if needed
output.parent.mkdir(parents=True, exist_ok=True)
print(f"\nCreated directory: {output.parent}")

# You Should See:
# Full path: data/output/2026/04/results.parquet
# Parent:    data/output/2026/04
# Stem:      results
# Suffix:    .parquet
# Name:      results.parquet
# Created directory: data/output/2026/04

## Recap

| Format | Read | Write | When |
|---|---|---|---|
| Text | `open(f).read()` | `open(f, 'w').write()` | Logs, SQL, config |
| CSV | `csv.DictReader` / `pd.read_csv` | `csv.DictWriter` / `df.to_csv` | Tabular exchange |
| JSON | `json.load(f)` | `json.dump(obj, f)` | API, config, NoSQL |
| JSON Lines | `[json.loads(line) for line in f]` | `json.dumps(obj) + '\n'` | Streaming, BigQuery |
| Parquet | `pd.read_parquet(f)` | `df.to_parquet(f)` | Data lake, analytics |

**Key rules:**
- Always use `with open()` — never leave files unclosed
- Use `pathlib.Path` — never string concatenation for paths
- Set `timeout` on every API call
- Use `response.raise_for_status()` to catch HTTP errors

**Next notebook:** [NumPy and Pandas](Python_NumPy_Pandas.ipynb)

**Full explanation:** [File I/O Chapter](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/python/04_File_IO.md)